# ComplaintIQ - logistic-regression baselines (`04b_baselines_logreg`)

Simple **learned** baselines for `monetary_relief`, one notch above the heuristics in
`04a_baselines_trivial.ipynb`. Still **no feature engineering** - raw columns, default
model params, no tuning. The point is to see whether an off-the-shelf logistic regression
beats the product-bucket heuristic bar (~36x / 19x / 9.7x lift at top-1% / 5% / 10% from `04a` / `02` section 9a).

Two baselines:
- **Metadata logistic regression**: one-hot of the raw intake categoricals, applies to
  every complaint.
- **TF-IDF logistic regression**: default TF-IDF over the raw narrative, on the ~23% of
  complaints that carry text (the other ~77% still need the metadata path - see the
  calibration note in `02` section 8).

## How to read this notebook
Same evaluation discipline as `04a`: **chronological split**, **proportional stratified
sample**, and the shared imbalance-aware metrics (PR-AUC, ROC-AUC, lift at top-1% / 5% / 10%, Brier).
Spark does the split and sampling; sklearn fits on the pandas sample. `class_weight=
"balanced"` is the one concession to imbalance - it is a model setting, not a feature.

> **Note:** the two models cover different populations (all complaints vs the narrative
> subset), so their metrics are reported against **their own** test base rate. Merging them
> into one queue is a calibration problem flagged in `02` section 8, not done here.

## Setup

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from pyspark.sql import DataFrame as SparkDataFrame
from typing import Any
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.metrics import report, top_k_lift
from complaintiq.sampling import stratified_pandas

# The tools we lean on across the project.
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn loaded")

---
## 1. Load + chronological split *(shared modeling foundation)*

Same split as `04a` (80th-percentile date cut), but keep `complaint_text` and
`has_narrative` too so the TF-IDF baseline has the raw narrative.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
path = data_dir / "complaints.parquet"
if not path.exists():
    raise FileNotFoundError("data/complaints.parquet not found. Run `make parquet` first.")

META_COLS = ["product", "sub_product", "issue", "sub_issue", "submitted_via", "state", "tags"]
spark_df = (
    spark.read.parquet(str(path))
    .select(
        "monetary_relief",
        "has_narrative",
        "complaint_text",
        F.to_date("date_received").alias("date_received"),
        *META_COLS,
    )
    .dropna(subset=["date_received"])
)

spark_df = spark_df.withColumn("epoch", F.datediff("date_received", F.lit("1970-01-01")))
cut = spark_df.approxQuantile("epoch", [0.80], 0.001)[0]
train_sdf = spark_df.filter(F.col("epoch") <= cut).drop("epoch")
test_sdf = spark_df.filter(F.col("epoch") > cut).drop("epoch")
print(f"train rows: {train_sdf.count():,}  |  test rows: {test_sdf.count():,}")

---
## 2. Stratified sample + shared helpers

Proportional per-class sample into pandas, plus the same `report()` scoreboard as `04a` so
every model here is comparable to the heuristics there.

In [ ]:
train = stratified_pandas(train_sdf, 300_000)
test = stratified_pandas(test_sdf, 300_000)
y_train = train["monetary_relief"].to_numpy()
y_test = test["monetary_relief"].to_numpy()
print(f"train sample: {len(train):,} (pos {y_train.mean():.4%})")
print(f"test sample:  {len(test):,} (pos {y_test.mean():.4%})")
results = []

---
## 3. Baseline C - logistic regression on raw metadata

One-hot the raw intake categoricals with `DictVectorizer` (missing values become an
explicit `MISSING` category, which is informative, not engineered) and fit a default
logistic regression with balanced class weights. Applies to **every** complaint.

In [ ]:
CAT_COLS = ["product", "sub_product", "issue", "sub_issue", "submitted_via", "state", "tags"]


def categorical_dict_records(frame: pd.DataFrame) -> list[dict[str, str]]:
    # Raw categoricals as strings; NaN -> "MISSING" so DictVectorizer gets clean keys.
    return frame[CAT_COLS].fillna("MISSING").astype(str).to_dict("records")


dict_vectorizer = DictVectorizer(sparse=True)
X_train = dict_vectorizer.fit_transform(categorical_dict_records(train))
X_test = dict_vectorizer.transform(categorical_dict_records(test))

meta_lr = LogisticRegression(max_iter=200, class_weight="balanced")
meta_lr.fit(X_train, y_train)
scores_meta = meta_lr.predict_proba(X_test)[:, 1]
results.append(report("logreg_metadata", y_test, scores_meta))

> **What you're seeing:** a default logistic regression on raw one-hot metadata.
>
> **Notice:** higher PR-AUC than the heuristic, but **lift** barely moves at any queue size (top-1% / 5% / 10%) - top-10% is capped near 10x, and the smaller queues do not separate either.
>
> **Why it matters:** proves the point of computing the baseline - a model can look better
> on PR-AUC yet barely beat the product-bucket heuristic on the metric the product cares
> about. That gap is the target for real feature work later.

---
## 4. Baseline D - logistic regression on raw TF-IDF narrative

Default `TfidfVectorizer` (`min_df=5`, unigrams, English stop words) over the raw narrative,
on the **subset of complaints that have text**. No custom tokenization or n-gram tuning.
Reported against the narrative subset's own base rate, since it only scores that population.

In [ ]:
train_txt = train[train["has_narrative"] & train["complaint_text"].notna()]
test_txt = test[test["has_narrative"] & test["complaint_text"].notna()]
y_train_narrative = train_txt["monetary_relief"].to_numpy()
y_test_narrative = test_txt["monetary_relief"].to_numpy()

tfidf = TfidfVectorizer(max_features=20_000, min_df=5, stop_words="english")
tfidf_train = tfidf.fit_transform(train_txt["complaint_text"])
tfidf_test = tfidf.transform(test_txt["complaint_text"])

text_lr = LogisticRegression(max_iter=200, class_weight="balanced")
text_lr.fit(tfidf_train, y_train_narrative)
scores_text = text_lr.predict_proba(tfidf_test)[:, 1]
print(
    f"narrative subset - train {len(train_txt):,}  test {len(test_txt):,}  "
    f"(subset base rate {y_test_narrative.mean():.4%})"
)
results.append(report("logreg_tfidf_text", y_test_narrative, scores_text))

> **What you're seeing:** default TF-IDF + logistic regression on the ~23% with a narrative.
>
> **Notice:** the strongest PR-AUC so far - text carries real signal - but on a denser
> subset (its base rate is higher than the overall 1.28%), so its lift is not directly
> comparable to the metadata models.
>
> **Why it matters:** confirms the narrative is worth modeling, and sets up the calibration
> problem from `02` section 8: to merge this text score with the metadata score in one queue,
> both must land on the same probability scale.

---
## 5. Baselines summary

In [ ]:
board = pd.DataFrame(results).set_index("model")
display(board.round(4))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(
    x=board["pr_auc"],
    y=board.index,
    hue=board.index,
    palette="colorblind",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("PR-AUC by baseline")
axes[0].set_xlabel("PR-AUC")
axes[0].set_ylabel("")
sns.barplot(
    x=board["lift"], y=board.index, hue=board.index, palette="colorblind", legend=False, ax=axes[1]
)
axes[1].set_title("Top-10% lift by baseline")
axes[1].set_xlabel("lift over base rate")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Persist per-model metrics to the volume so they are retrievable outside the run
# (the jobs API does not expose notebook stdout). Mirrors the pattern in 07a-10.
# No-op locally when dbutils is absent. report() already returns lift at 1/5/10%.
try:
    import json as _json
    import time as _time

    _payload = {
        "notebook": "04b_baselines_logreg",
        "test_base_rate": float(y_test.mean()),
        "test_base_rate_narrative": float(y_test_narrative.mean()),
        "results": results,
        "ts": _time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    dbutils.fs.put(  # noqa: F821 - Databricks-injected global
        "/Volumes/workspace/complaintiq/data/metrics_04b.json",
        _json.dumps(_payload, indent=2),
        overwrite=True,
    )
    print("wrote /Volumes/workspace/complaintiq/data/metrics_04b.json")
except NameError:
    print("dbutils unavailable (local run); skipped metrics persistence")

---
## 6. Takeaways

> **What these baselines establish:**
> - **Metadata logistic regression** lifts PR-AUC over the heuristic but barely beats its
>   lift at every queue size (top-1% / 5% / 10%) - the honest gap real feature work has to close.
> - **TF-IDF text** is the strongest signal, but only on the narrative subset and on a
>   different base rate, so it is not a like-for-like queue yet.
> - **The open problem** is not a new model but **calibrating** the metadata and text scores
>   onto one probability scale so a single ranked queue is fair (`02` section 8).

These are the numbers to beat with actual feature engineering and model iteration.
Unsupervised baselines are in **`05_baselines_unsupervised.ipynb`**.